# Chapter 34: Visual Odometry vs SLAM

<a href="../lite/lab/index.html?path=ch34_visual_odometry.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ── Rotation helpers ─────────────────────────────────────────────────────────
def Rz(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def Ry(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

# ── Pinhole projection ───────────────────────────────────────────────────────
def project(K, R, t, pts_world):
    \"\"\"Project 3D world points to 2D pixels given camera pose (R, t).
    Camera sees points in its frame: p_cam = R @ (p_world - t)
    Then projects: pixel = K @ p_cam / p_cam_z
    \"\"\"
    pts_cam = (R @ (pts_world - t).T).T  # Nx3
    valid = pts_cam[:, 2] > 0.1
    pixels = np.full((len(pts_world), 2), np.nan)
    if valid.any():
        proj = (K @ pts_cam[valid].T).T
        pixels[valid] = proj[:, :2] / proj[:, 2:3]
    return pixels, valid

# ── Essential matrix from R, t ───────────────────────────────────────────────
def skew(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])

def essential_from_Rt(R, t):
    return skew(t) @ R

# ── Triangulation (DLT) ─────────────────────────────────────────────────────
def triangulate_points(K, R1, t1, R2, t2, pixels1, pixels2):
    \"\"\"Triangulate 3D points from two views using DLT.\"\"\"
    P1 = K @ np.hstack([R1, -R1 @ t1.reshape(3,1)])
    P2 = K @ np.hstack([R2, -R2 @ t2.reshape(3,1)])
    pts_3d = []
    for p1, p2 in zip(pixels1, pixels2):
        A = np.array([
            p1[0] * P1[2] - P1[0],
            p1[1] * P1[2] - P1[1],
            p2[0] * P2[2] - P2[0],
            p2[1] * P2[2] - P2[1]
        ])
        _, _, Vt = np.linalg.svd(A)
        X = Vt[-1]
        pts_3d.append(X[:3] / X[3])
    return np.array(pts_3d)

# ── Pose estimation from essential matrix (8-point algorithm simplified) ─────
def estimate_pose_from_correspondences(K, pts1_px, pts2_px):
    \"\"\"Estimate R, t from pixel correspondences using the 8-point algorithm.
    Returns R, t (unit translation direction).\"\"\"
    K_inv = np.linalg.inv(K)
    # Normalize to camera coordinates
    p1 = (K_inv @ np.hstack([pts1_px, np.ones((len(pts1_px), 1))]).T).T
    p2 = (K_inv @ np.hstack([pts2_px, np.ones((len(pts2_px), 1))]).T).T

    # Build the constraint matrix for E
    n = len(p1)
    A = np.zeros((n, 9))
    for i in range(n):
        x1, y1 = p1[i, 0], p1[i, 1]
        x2, y2 = p2[i, 0], p2[i, 1]
        A[i] = [x2*x1, x2*y1, x2, y2*x1, y2*y1, y2, x1, y1, 1]

    _, _, Vt = np.linalg.svd(A)
    E = Vt[-1].reshape(3, 3)

    # Enforce rank-2 constraint
    U, S, Vt2 = np.linalg.svd(E)
    S_new = np.array([(S[0]+S[1])/2, (S[0]+S[1])/2, 0])
    E = U @ np.diag(S_new) @ Vt2

    # Decompose E into R, t (4 solutions, pick the one with most points in front)
    W = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]])
    R_candidates = [U @ W @ Vt2, U @ W.T @ Vt2]
    t_candidates = [U[:, 2], -U[:, 2]]

    best_count = -1
    best_R, best_t = None, None
    for R_c in R_candidates:
        if np.linalg.det(R_c) < 0:
            R_c = -R_c
        for t_c in t_candidates:
            # Check how many points are in front of both cameras
            count = 0
            for i in range(n):
                p3d = np.cross(p1[i], R_c @ p2[i] + t_c)
                # Simple depth check
                pt_cam1 = p1[i] * 5  # rough depth
                if pt_cam1[2] > 0:
                    count += 1
            if count > best_count:
                best_count = count
                best_R = R_c
                best_t = t_c / (np.linalg.norm(t_c) + 1e-10)

    return best_R, best_t

Visual Odometry says "I moved THIS much since my last frame." Visual SLAM says
"I am HERE in the world." The difference is a map.

VO is like counting your steps with your eyes closed between peeks. SLAM is like
drawing a map as you walk and checking it constantly. VO drifts. SLAM corrects.

In this chapter we will build a complete visual odometry pipeline from scratch:
detect features in each camera frame, match them across frames, estimate the
relative pose, and chain those poses into a trajectory. Then we will watch it
drift, and understand why SLAM is needed.

## How Visual Odometry Works

```
Frame t          Frame t+1
┌─────────┐      ┌─────────┐
│  *   *  │      │   *  *  │     1. Detect features in both frames
│ *  *    │ ───► │  *  *   │     2. Match features across frames
│   *   * │      │    * *  │     3. Estimate relative pose (R, t)
└─────────┘      └─────────┘     4. Chain: T_world = T_world @ T_relative
```

```{admonition} What you will build
:class: tip

- Build a complete visual odometry pipeline: project, match, estimate pose, chain
- Watch VO drift accumulate over a 40 frame trajectory
- Compare VO (drifts) with SLAM (corrects at loop closure)
- Diagnose VO failure from pure rotation (degenerate optical flow)

**Real world application:** Visual odometry runs on Mars rovers, drones, and AR headsets. After this chapter, you will understand its power (works with just a camera) and its fundamental limitation (drift).
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **libviso2** | Efficient stereo visual odometry library |
| **ORB-SLAM3** | State of the art visual(-inertial) odometry and SLAM |
| **OpenCV solvePnP** | Pose estimation from 3D to 2D correspondences (used in VO tracking) |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 34.1 Frame to Frame Estimation

The core of VO: given two camera frames showing the same scene from slightly different
viewpoints, estimate how the camera moved between them.

**The pipeline:**
1. **Detect** feature points in both images (corners, blobs)
2. **Match** features by comparing descriptors (or track with optical flow)
3. **Estimate** the essential matrix $E$ from the matched pixel coordinates
4. **Decompose** $E$ into relative rotation $R$ and translation direction $\hat{t}$
5. **Triangulate** 3D points to recover the translation scale (or use another source)

Let's simulate this with a 3D scene and a moving camera.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_landmarks = 50          # number of 3D points in the scene
scene_depth = (5, 20)     # depth range of scene points (meters)
scene_width = 10          # horizontal spread
fx, fy = 500, 500         # focal length (pixels)
cx, cy = 320, 240         # principal point
pixel_noise = 0.5         # feature detection noise (pixels)  (try 0, 0.5, 2, 5)

# Camera motion: small forward + rightward movement with slight rotation
move_forward = 0.5        # meters forward (Z)   (try 0.1, 0.5, 2.0)
move_right = 0.3          # meters right (X)
yaw_deg = 2.0             # rotation about Y axis (degrees)
# ──────────────────────────────────────────────────────────────────────────────

K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

# Generate 3D scene (random points in front of camera)
landmarks = np.column_stack([
    np.random.uniform(-scene_width, scene_width, n_landmarks),
    np.random.uniform(-3, 3, n_landmarks),
    np.random.uniform(*scene_depth, n_landmarks)
])

# Camera 1 at origin
R1 = np.eye(3)
t1 = np.zeros(3)

# Camera 2: moved and rotated
R2 = Ry(np.radians(yaw_deg))
t2 = np.array([move_right, 0, move_forward])

# Project to both frames
pix1, valid1 = project(K, R1, t1, landmarks)
pix2, valid2 = project(K, R2, t2, landmarks)

# Features visible in BOTH frames and within image bounds
img_w, img_h = 640, 480
in_bounds1 = valid1 & (pix1[:, 0] >= 0) & (pix1[:, 0] < img_w) & (pix1[:, 1] >= 0) & (pix1[:, 1] < img_h)
in_bounds2 = valid2 & (pix2[:, 0] >= 0) & (pix2[:, 0] < img_w) & (pix2[:, 1] >= 0) & (pix2[:, 1] < img_h)
both_visible = in_bounds1 & in_bounds2 & np.all(np.isfinite(pix1), axis=1) & np.all(np.isfinite(pix2), axis=1)

# Add pixel noise (simulates imperfect feature detection)
pix1_noisy = pix1.copy()
pix2_noisy = pix2.copy()
pix1_noisy[both_visible] += np.random.normal(0, pixel_noise, (both_visible.sum(), 2))
pix2_noisy[both_visible] += np.random.normal(0, pixel_noise, (both_visible.sum(), 2))

# ── Visualize the two "frames" ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Frame 1
ax = axes[0]
ax.scatter(pix1_noisy[both_visible, 0], pix1_noisy[both_visible, 1],
           c='steelblue', s=20, zorder=5)
ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
ax.set_title(f"Frame 1: {both_visible.sum()} features detected", fontsize=13)
ax.set_xlabel("u (px)"); ax.set_ylabel("v (px)")

# Frame 2
ax = axes[1]
ax.scatter(pix2_noisy[both_visible, 0], pix2_noisy[both_visible, 1],
           c='tomato', s=20, zorder=5)
ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
ax.set_title(f"Frame 2: same features, different positions", fontsize=13)
ax.set_xlabel("u (px)")

# Optical flow: show feature motion between frames
ax = axes[2]
for i in np.where(both_visible)[0][:30]:  # show 30 tracks for clarity
    ax.annotate("", xy=pix2_noisy[i], xytext=pix1_noisy[i],
                arrowprops=dict(arrowstyle='->', color='forestgreen', lw=0.8, alpha=0.7))
ax.scatter(pix1_noisy[both_visible, 0][:30], pix1_noisy[both_visible, 1][:30],
           c='steelblue', s=15, zorder=5, label='frame 1')
ax.scatter(pix2_noisy[both_visible, 0][:30], pix2_noisy[both_visible, 1][:30],
           c='tomato', s=15, zorder=5, label='frame 2')
ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
ax.set_title("Feature motion (optical flow)", fontsize=13)
ax.set_xlabel("u (px)"); ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Scene: {n_landmarks} 3D points, {both_visible.sum()} visible in both frames")
print(f"Camera moved: [{move_right:.1f}, 0, {move_forward:.1f}]m, rotated {yaw_deg:.1f}° yaw")

**Key observations:**
- Each "frame" is just a set of 2D pixel coordinates of detected features.
- The **optical flow** (green arrows) shows how features moved between frames.
- The flow pattern encodes the camera's motion: forward motion creates an expanding pattern from the center (focus of expansion); rotation creates a uniform shift.
- Pixel noise makes pose estimation harder. Try increasing `pixel_noise` to see the effect.

### Estimating the relative pose

From the matched pixel coordinates, we estimate the **essential matrix** $E$ using
the 8-point algorithm, then decompose it into rotation $R$ and translation direction $\hat{t}$.

In [ ]:
# ── Estimate pose from the noisy correspondences ─────────────────────────────
matched_px1 = pix1_noisy[both_visible]
matched_px2 = pix2_noisy[both_visible]

# Use our 8-point algorithm implementation
R_est, t_est = estimate_pose_from_correspondences(K, matched_px1, matched_px2)

# True relative pose
R_rel_true = R2 @ R1.T
t_rel_true = t2 - t1
t_dir_true = t_rel_true / np.linalg.norm(t_rel_true)

# Compare
angle_err = np.degrees(np.arccos(np.clip((np.trace(R_est.T @ R_rel_true) - 1) / 2, -1, 1)))
t_dot = abs(np.dot(t_est, t_dir_true))

print("═" * 55)
print("  Relative Pose Estimation Result")
print("═" * 55)
print(f"\n  True R (first row):  {R_rel_true[0]}")
print(f"  Est. R (first row):  {R_est[0]}")
print(f"  Rotation error:      {angle_err:.3f}°")
print(f"\n  True t direction:    {t_dir_true}")
print(f"  Est. t direction:    {t_est}")
print(f"  Direction agreement: {t_dot:.4f}  (1.0 = perfect)")
print(f"\n  NOTE: Translation scale is UNKNOWN from vision alone!")

## 34.2 Drift

Now comes the critical part: we chain relative poses over many frames to build
a trajectory. Each frame to frame estimate has a small error. These errors **accumulate**.

Let's simulate a camera driving along a known path, estimating the pose between
every consecutive pair of frames, and watch the estimated trajectory drift away
from the truth.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_frames = 40             # number of camera frames  (try 20, 40, 80)
step_size = 0.5           # forward motion per frame (meters)
turn_rate_deg = 5.0       # degrees of yaw per frame (circular trajectory)
pixel_noise_vo = 1.0      # pixel noise  (try 0.5, 1.0, 3.0)
n_scene_points = 100      # 3D points in the world
# ──────────────────────────────────────────────────────────────────────────────

# Generate a larger scene
scene_pts = np.column_stack([
    np.random.uniform(-15, 15, n_scene_points),
    np.random.uniform(-3, 3, n_scene_points),
    np.random.uniform(2, 25, n_scene_points)
])

# Ground truth camera trajectory (circular path)
gt_poses = []  # list of (R, t)
R_cur = np.eye(3)
t_cur = np.zeros(3)
gt_poses.append((R_cur.copy(), t_cur.copy()))

for i in range(1, n_frames):
    # Move forward in camera's Z direction, then rotate
    forward = R_cur.T @ np.array([0, 0, step_size])
    t_cur = t_cur + forward
    R_cur = Ry(np.radians(turn_rate_deg)) @ R_cur
    gt_poses.append((R_cur.copy(), t_cur.copy()))

# ── Run Visual Odometry: estimate pose between consecutive frames ────────────
vo_R = np.eye(3)
vo_t = np.zeros(3)
vo_trajectory = [vo_t.copy()]
gt_trajectory = [np.zeros(3)]

for i in range(1, n_frames):
    R_prev, t_prev = gt_poses[i-1]
    R_curr, t_curr = gt_poses[i]

    # Project scene into both frames
    px_prev, v_prev = project(K, R_prev, t_prev, scene_pts)
    px_curr, v_curr = project(K, R_curr, t_curr, scene_pts)

    # Find common visible features in image bounds
    in_img = lambda px, v: v & (px[:,0]>=0) & (px[:,0]<640) & (px[:,1]>=0) & (px[:,1]<480) & np.all(np.isfinite(px), axis=1)
    vis_prev = in_img(px_prev, v_prev)
    vis_curr = in_img(px_curr, v_curr)
    common = vis_prev & vis_curr

    if common.sum() >= 8:
        # Add noise
        p1 = px_prev[common] + np.random.normal(0, pixel_noise_vo, (common.sum(), 2))
        p2 = px_curr[common] + np.random.normal(0, pixel_noise_vo, (common.sum(), 2))

        try:
            R_est_i, t_est_i = estimate_pose_from_correspondences(K, p1, p2)
            # Scale: use ground truth scale (in real VO, this comes from stereo, IMU, or known objects)
            true_scale = np.linalg.norm(t_curr - t_prev)
            t_est_scaled = t_est_i * true_scale

            # Chain: update VO pose
            vo_t = vo_t + vo_R.T @ t_est_scaled
            vo_R = R_est_i @ vo_R
        except:
            # If estimation fails, use ground truth (skip this frame)
            true_R_rel = R_curr @ R_prev.T
            true_t_rel = t_curr - t_prev
            vo_t = vo_t + vo_R.T @ (vo_R @ true_t_rel)
            vo_R = true_R_rel @ vo_R
    else:
        # Not enough features, dead reckon with ground truth
        true_R_rel = R_curr @ R_prev.T
        vo_t = vo_t + R_prev.T @ (t_curr - t_prev)

    vo_trajectory.append(vo_t.copy())
    gt_trajectory.append(t_curr.copy())

vo_trajectory = np.array(vo_trajectory)
gt_trajectory = np.array(gt_trajectory)

# ── Visualize ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Top-down view (X-Z plane)
ax = axes[0]
ax.plot(gt_trajectory[:, 0], gt_trajectory[:, 2], 'k-', lw=2, label='ground truth')
ax.plot(vo_trajectory[:, 0], vo_trajectory[:, 2], 'tomato', lw=2, marker='o', ms=3, label='visual odometry')
ax.plot(gt_trajectory[0, 0], gt_trajectory[0, 2], 'go', ms=12, zorder=10, label='start')
ax.plot(gt_trajectory[-1, 0], gt_trajectory[-1, 2], 'rs', ms=10, zorder=10, label='end (GT)')
ax.set_xlabel("X (m)"); ax.set_ylabel("Z (m)")
ax.set_title("Top down view: VO trajectory drifts!", fontsize=14)
ax.legend(); ax.set_aspect('equal')

# Error over time
ax = axes[1]
errors = np.linalg.norm(vo_trajectory - gt_trajectory, axis=1)
ax.plot(errors, 'tomato', lw=2, marker='o', ms=3)
ax.set_xlabel("Frame number"); ax.set_ylabel("Position error (m)")
ax.set_title(f"Drift grows over time (final error: {errors[-1]:.2f}m)", fontsize=13)
ax.axhline(0, color='k', ls='--', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Frames: {n_frames}, Pixel noise: {pixel_noise_vo}px")
print(f"Final drift: {errors[-1]:.3f}m after {n_frames * step_size:.1f}m traveled")
print(f"Relative drift: {100 * errors[-1] / (n_frames * step_size):.1f}% of distance traveled")

**Key observations:**
- VO drift is **inevitable**. Each frame to frame estimate has a small error, and these errors compound.
- The drift is roughly proportional to the distance traveled (typically 0.5% to 5% for good VO systems).
- **Pixel noise** directly affects drift rate. Try changing `pixel_noise_vo` from 0.5 to 3.0.
- VO gives you **relative** motion. It does not know where you are in the world.
- This is precisely why we need SLAM: it adds **loop closure** to correct the accumulated drift.

## 34.3 Role of Mapping

The difference between VO and SLAM is the **map**. VO chains relative poses and forgets.
SLAM maintains a map of landmarks and uses it as an absolute reference.

When the robot sees a previously mapped landmark, it can correct its position estimate.
When it completes a loop back to the start, the accumulated drift gets distributed across
the entire trajectory.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_poses = 30
drift_per_step = 0.15       # VO drift per step (meters)
loop_closure_at = 29        # step where loop closure happens
loop_closure_error = 0.1    # loop closure measurement noise
# ──────────────────────────────────────────────────────────────────────────────

# Ground truth: circular trajectory
angles = np.linspace(0, 2*np.pi, n_poses, endpoint=False)
radius = 8.0
gt = np.column_stack([radius*np.cos(angles), radius*np.sin(angles)])

# VO: accumulates drift
vo = np.zeros((n_poses, 2))
vo[0] = gt[0]
for i in range(1, n_poses):
    step = gt[i] - gt[i-1]
    noise = np.random.normal(0, drift_per_step, 2)
    vo[i] = vo[i-1] + step + noise

# SLAM with loop closure: correct the drift
slam = vo.copy()
if loop_closure_at < n_poses:
    # Loop closure: pose[loop_closure_at] should be near pose[0]
    loop_error = slam[loop_closure_at] - slam[0]
    # Distribute correction linearly across all poses
    for i in range(1, loop_closure_at + 1):
        fraction = i / loop_closure_at
        slam[i] -= fraction * loop_error

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(gt[:, 0], gt[:, 1], 'k--', lw=1.5, label='ground truth')
ax.plot(gt[:, 0], gt[:, 1], 'k.', ms=4)
ax.set_title("Ground truth (circle)", fontsize=13)
ax.set_aspect('equal'); ax.legend()

ax = axes[1]
ax.plot(gt[:, 0], gt[:, 1], 'k--', lw=1, alpha=0.3)
ax.plot(vo[:, 0], vo[:, 1], 'tomato', lw=2, marker='o', ms=4, label='VO (drifts)')
ax.set_title(f"Visual Odometry: gap = {np.linalg.norm(vo[-1]-vo[0]):.2f}m", fontsize=13)
ax.set_aspect('equal'); ax.legend()

ax = axes[2]
ax.plot(gt[:, 0], gt[:, 1], 'k--', lw=1, alpha=0.3)
ax.plot(slam[:, 0], slam[:, 1], 'steelblue', lw=2, marker='o', ms=4, label='SLAM (corrected)')
ax.annotate("loop closure!", xy=slam[0], xytext=(slam[0,0]+2, slam[0,1]+2),
            arrowprops=dict(arrowstyle='->', color='forestgreen', lw=2),
            fontsize=12, color='forestgreen', fontweight='bold')
ax.set_title(f"SLAM with loop closure: closed!", fontsize=13)
ax.set_aspect('equal'); ax.legend()

plt.suptitle("VO drifts. SLAM corrects.", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

errors_vo = np.linalg.norm(vo - gt, axis=1)
errors_slam = np.linalg.norm(slam - gt, axis=1)
print(f"VO  mean error: {errors_vo.mean():.3f}m, max: {errors_vo.max():.3f}m")
print(f"SLAM mean error: {errors_slam.mean():.3f}m, max: {errors_slam.max():.3f}m")

## 34.4 State Representation

In VO, the state is just the current camera pose: $(R, t)$ or equivalently $(x, y, z, q_w, q_x, q_y, q_z)$.

In SLAM, the state includes the map:

$$\mathbf{x}_{VO} = \begin{bmatrix} R_t \\ t_t \end{bmatrix} \qquad \mathbf{x}_{SLAM} = \begin{bmatrix} R_t \\ t_t \\ l_1 \\ l_2 \\ \vdots \\ l_N \end{bmatrix}$$

The SLAM state grows with each new landmark. This is both its strength (it maintains
a map for loop closure and re-localization) and its weakness (computational cost grows).

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_frames_sim = 50
landmarks_per_frame = 5    # new landmarks observed per frame
# ──────────────────────────────────────────────────────────────────────────────

vo_state_size = np.ones(n_frames_sim) * 6  # always 6 (3 position + 3 orientation)
slam_state_size = np.zeros(n_frames_sim)
total_landmarks = 0
for i in range(n_frames_sim):
    total_landmarks += landmarks_per_frame * (1 if i < 20 else 0.2)  # fewer new landmarks over time
    slam_state_size[i] = 6 + 3 * int(total_landmarks)  # 6 pose + 3 per landmark

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(vo_state_size, 'steelblue', lw=3, label='VO state size (constant)')
ax.plot(slam_state_size, 'tomato', lw=3, label='SLAM state size (grows)')
ax.set_xlabel("Frame number"); ax.set_ylabel("State vector dimension")
ax.set_title("VO has constant cost. SLAM state grows with the map.", fontsize=13)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 34.5 Failure Modes

Visual odometry fails when the fundamental assumptions break down:

| Failure mode | Why it happens | What you see |
|---|---|---|
| **Pure rotation** | No translation → no triangulation → no scale | Infinite depth estimates, NaN |
| **Textureless scenes** | No features to detect | Zero or few matches |
| **Motion blur** | Fast motion smears features | Matches are wrong |
| **Occlusion** | Objects appear/disappear between frames | Outlier matches |
| **Degenerate motion** | Forward motion only → all flow radiates from one point | Poor conditioning |

### Demo: Pure rotation failure

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
yaw_only_deg = 5.0         # pure rotation, no translation
n_pts_demo = 40
# ──────────────────────────────────────────────────────────────────────────────

pts_3d_demo = np.column_stack([
    np.random.uniform(-5, 5, n_pts_demo),
    np.random.uniform(-3, 3, n_pts_demo),
    np.random.uniform(5, 15, n_pts_demo)
])

R_pure_rot = Ry(np.radians(yaw_only_deg))
t_zero = np.zeros(3)

px1_rot, v1_rot = project(K, np.eye(3), t_zero, pts_3d_demo)
px2_rot, v2_rot = project(K, R_pure_rot, t_zero, pts_3d_demo)

common_rot = v1_rot & v2_rot & np.all(np.isfinite(px1_rot), axis=1) & np.all(np.isfinite(px2_rot), axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for i in np.where(common_rot)[0]:
    ax.annotate("", xy=px2_rot[i], xytext=px1_rot[i],
                arrowprops=dict(arrowstyle='->', color='tomato', lw=0.8, alpha=0.6))
ax.scatter(px1_rot[common_rot, 0], px1_rot[common_rot, 1], c='steelblue', s=15)
ax.set_xlim(0, 640); ax.set_ylim(480, 0)
ax.set_title("Pure rotation: all flow is horizontal (no depth info!)", fontsize=12)

ax = axes[1]
# Show that with normal motion (translation + rotation), flow has structure
R_normal = Ry(np.radians(yaw_only_deg))
t_normal = np.array([0.3, 0, 0.5])
px2_norm, v2_norm = project(K, R_normal, t_normal, pts_3d_demo)
common_norm = v1_rot & v2_norm & np.all(np.isfinite(px1_rot), axis=1) & np.all(np.isfinite(px2_norm), axis=1)
for i in np.where(common_norm)[0]:
    ax.annotate("", xy=px2_norm[i], xytext=px1_rot[i],
                arrowprops=dict(arrowstyle='->', color='forestgreen', lw=0.8, alpha=0.6))
ax.scatter(px1_rot[common_norm, 0], px1_rot[common_norm, 1], c='steelblue', s=15)
ax.set_xlim(0, 640); ax.set_ylim(480, 0)
ax.set_title("Translation + rotation: flow has depth-dependent structure", fontsize=12)

plt.suptitle("Pure rotation kills visual odometry", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Pure rotation: all flow vectors are the same length → no depth information")
print("With translation: nearby points move more than far points → depth is recoverable")

**Key observations:**
- **Pure rotation** is the most common VO failure. It happens whenever the robot turns in place.
- The essential matrix becomes degenerate (rank deficient) with pure rotation.
- Solutions: detect pure rotation and skip those frames, or use IMU to separate rotation from translation.
- **Textureless scenes** (white walls, clear sky) are the second most common failure.

---

## Exercises

### Exercise 34.1: Pixel noise vs pose accuracy

Run the two-frame pose estimation (Section 34.1) with pixel noise values
of [0, 0.5, 1, 2, 5, 10]. For each, compute the rotation and translation direction error.
Plot both errors vs noise level. At what noise level does VO become unreliable?

In [ ]:
# Your code here
noise_levels = [0, 0.5, 1, 2, 5, 10]
rot_errors = []
trans_errors = []
# For each noise level:
#   Generate noisy correspondences
#   Estimate pose
#   Compute rotation and translation error
# Plot both vs noise level

### Exercise 34.2: Trajectory length vs drift

Run the VO trajectory simulation (Section 34.2) for trajectory lengths of
[10, 20, 40, 80, 160] frames. Plot the final position error vs number of frames.
Is the relationship linear, quadratic, or something else?

In [ ]:
# Your code here
frame_counts = [10, 20, 40, 80, 160]
# For each: run VO, record final error
# Plot and fit a curve

### Exercise 34.3: Number of features vs accuracy

In Section 34.1, vary the number of scene landmarks from 10 to 200.
For each, compute the pose estimation error. How many features do you
need for reliable pose estimation?

In [ ]:
# Your code here

### Exercise 34.4: Build a full VO pipeline (capstone)

Build a complete visual odometry system:
1. Generate a figure-8 trajectory (40 frames)
2. Create a 3D scene with 200 random points
3. For each pair of consecutive frames:
   a. Project points to both frames
   b. Add pixel noise
   c. Estimate relative pose (8-point algorithm)
   d. Chain the pose
4. Plot: ground truth vs VO trajectory (top down)
5. Plot: position error over time
6. Report: drift as percentage of distance traveled

In [ ]:
# Your code here
# This is the full VO pipeline! Take your time with this one.
# Use the functions defined in the setup cell.

# Step 1: Generate figure-8 trajectory
n_frames_capstone = 40
t_param = np.linspace(0, 2*np.pi, n_frames_capstone, endpoint=False)
# figure-8: x = sin(t), z = sin(2t)/2 + offset
gt_x = 5 * np.sin(t_param)
gt_z = 3 * np.sin(2 * t_param) + 10

# Step 2: Create scene
# Step 3-4: Run VO frame by frame
# Step 5-6: Evaluate